In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim 

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets & DataLoader

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Image => Scale(0,1) => Normalize(-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
testset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

Build CNN

In [10]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32, kernel_size=2, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernal Size = 2, Stride = 2

            nn.Conv2d(32,64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # falatten 
        x = self.fc_layers(x)

        return x

In [11]:
model = CNN()

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [16]:
# Train The CNN

epochs = 10 

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        outputs = model.forward(images) # FP
        loss = criterion(outputs, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # Update Params

        epoch_training_loss += loss.item()

    print(f"epoch {epoch+1} / {epochs} & loss = {epoch_training_loss/ len(trainloader)} ")

epoch 1 / 10 & loss = 0.06539783380149156 
epoch 2 / 10 & loss = 0.06130816952941422 
epoch 3 / 10 & loss = 0.058422410691840586 
epoch 4 / 10 & loss = 0.06149416807500045 
epoch 5 / 10 & loss = 0.05413307592301341 
epoch 6 / 10 & loss = 0.053907579838131825 
epoch 7 / 10 & loss = 0.05551319983129835 
epoch 8 / 10 & loss = 0.04998392866932742 
epoch 9 / 10 & loss = 0.05075432454287063 
epoch 10 / 10 & loss = 0.05640371009597883 


In [18]:
# Evaluate the CNN model

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        output = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (correct_labels == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy: {correct_labels / total_labels * 100} ")

Accuracy: 0.13999999999999999 
